# Data Loading for a GPT-style Language Model

Before we can train a language model, we need to turn raw text into batches of `(input, target)` tensors that a PyTorch model can consume. This notebook covers the **data pipeline** end-to-end: loading a text corpus, tokenizing it with GPT-2's Byte Pair Encoding, wrapping it in a sliding-window `Dataset`, splitting into train and validation sets, and producing PyTorch `DataLoader`s ready for a training loop.

### What we will cover

1. **Loading** the corpus (`the-verdict.txt`)
2. **Tokenizing** with GPT-2 BPE via `tiktoken`
3. **Sliding-window dataset** (`GPTDatasetV1`) producing shifted `(input, target)` pairs
4. **Train / validation split** by position in the text
5. **`DataLoader`** wrappers and a look at one batch

---

## 1. Setup

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))
print("numpy version:", version("numpy"))


torch version: 1.11.0
tiktoken version: 0.12.0
numpy version: 1.26.4


In [2]:
import os
from pathlib import Path

import numpy as np
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)
np.random.seed(123)


---

## 2. Loading the Corpus

We use **the-verdict.txt** from Chapter 1 (`01_llm_tokenizer/`). It is a short, self-contained text — perfect for demonstrating training end-to-end on a laptop. A *real* language model is trained on hundreds of billions of tokens; we will train on roughly 5,000.

In [3]:
# Locate the corpus — the-verdict.txt lives in the tokenizer notebook folder.
CORPUS_PATH = Path("../01_llm_tokenizer/the-verdict.txt")
assert CORPUS_PATH.exists(), f"Corpus not found at {CORPUS_PATH.resolve()}"

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total characters: {len(raw_text):,}")
print(f"Total words:      {len(raw_text.split()):,}")
print()
print("First 300 characters:")
print(raw_text[:300])

Total characters: 20,479
Total words:      3,634

First 300 characters:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would ha


---

## 3. Tokenization (GPT-2 BPE)

We tokenize with **GPT-2's Byte Pair Encoding** via `tiktoken` — the same tokenizer used by real GPT-2. This gives us a fixed vocabulary of 50,257 subword tokens, so we never need an `<unk>` token.

The model we train only ever *uses* a tiny fraction of this vocabulary (the words that appear in our small corpus), but the output layer still produces logits over all 50,257 tokens — the model learns to put almost all the probability mass on the subset it has actually seen.

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = tokenizer.encode(raw_text)
print(f"Vocabulary size:  {tokenizer.n_vocab:,}")
print(f"Number of tokens: {len(token_ids):,}")
print()
print("First 20 token IDs:", token_ids[:20])
print("Decoded back:       ", repr(tokenizer.decode(token_ids[:20])))

Vocabulary size:  50,257
Number of tokens: 5,145

First 20 token IDs: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]
Decoded back:        'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--'


---

## 4. Batching with a Sliding-Window Dataset

Language models are trained on `(input, target)` pairs where the target is the input **shifted by one position**. Each position in the input has a next-token target; the model predicts every position simultaneously (the causal mask ensures position *i* only uses information from positions *0..i*).

We build a standard **sliding-window dataset** (`GPTDatasetV1` from Chapter 1): chunk the token stream into windows of length `max_length`, each shifted from the previous by `stride` tokens.

In [5]:
class GPTDatasetV1(Dataset):
    """Sliding-window next-token-prediction dataset (from Chapter 1)."""

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Need at least max_length+1 tokens"

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk  = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

### 4.1 Train / Validation Split

A held-out validation set lets us detect **overfitting** — when the model memorizes the training text rather than learning general patterns. We split by **position in the text** (first 90% train, last 10% val) rather than randomly, since we want each split to contain contiguous prose.

In [6]:
TRAIN_RATIO = 0.90
split_idx = int(TRAIN_RATIO * len(raw_text))

train_text = raw_text[:split_idx]
val_text   = raw_text[split_idx:]

print(f"Train characters: {len(train_text):,}")
print(f"Val   characters: {len(val_text):,}")

Train characters: 18,431
Val   characters: 2,048


In [7]:
def create_dataloader(text, tokenizer, batch_size, max_length,
                       stride, shuffle, drop_last=True):
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=0,
    )

# Hyperparameters for the data pipeline
CONTEXT_LEN = 128     # tokens per training example (the model's context window)
STRIDE      = 128     # non-overlapping windows for train (stride == context_len)
BATCH_SIZE  = 2       # small — our corpus is tiny

train_loader = create_dataloader(
    train_text, tokenizer,
    batch_size=BATCH_SIZE, max_length=CONTEXT_LEN,
    stride=STRIDE, shuffle=True, drop_last=True,
)
val_loader = create_dataloader(
    val_text, tokenizer,
    batch_size=BATCH_SIZE, max_length=CONTEXT_LEN,
    stride=STRIDE, shuffle=False, drop_last=False,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

# Peek at one batch to confirm shapes
x_batch, y_batch = next(iter(train_loader))
print(f"\nInput shape:  {tuple(x_batch.shape)}  (batch, context_len)")
print(f"Target shape: {tuple(y_batch.shape)}  (batch, context_len)")
print(f"Target is input shifted by 1: {torch.equal(x_batch[0, 1:], y_batch[0, :-1])}")

Train batches: 18
Val   batches: 2

Input shape:  (2, 128)  (batch, context_len)
Target shape: (2, 128)  (batch, context_len)
Target is input shifted by 1: True
